In [ ]:
import sys
print(sys.executable)

In [ ]:
import nflreadpy as nfl
help(nfl)

In [ ]:
next_gen = nfl.load_nextgen_stats(2025)
print(next_gen.columns)

In [ ]:
advstats = nfl.load_pfr_advstats(2025)
print(advstats.columns)

In [ ]:
pbr = nfl.load_pbp(2025)
print(pbr.columns)

In [ ]:
import polars as pl

In [ ]:
df = nfl.load_player_stats([2025])

In [ ]:
print(df.head())

In [ ]:
print(df.columns)

In [ ]:
# with pd.option_context('display.max_rows', None):
#     print(df.filter(player_name='A.Rodgers')[['player_name','team', 'week', 'passing_yards', 'rushing_yards', 'position', 'position_group']])

In [ ]:

# This temporarily removes the row limit for just this block
with pl.Config(tbl_rows=-1):
    print(
        df.head()
        .select(['player_id', 'player_name', 'team', 'week', 'passing_yards', 'rushing_yards', 'fantasy_points', 'position', 'position_group'])
    )


In [ ]:
snap = nfl.load_snap_counts([2023, 2024, 2025])
print(snap.shape)
print(snap.filter((pl.col('position') == 'TE') & (pl.col('player') == 'Dalton Kincaid')).select(['pfr_player_id', 'player', 'team', 'week', 'offense_snaps', 'defense_snaps']))

In [ ]:
print(snap.columns)

In [ ]:
roster = nfl.load_rosters([2026])

In [ ]:
print(roster.columns)

In [ ]:
print(roster.filter((pl.col('team') == 'MIN') & (pl.col('position') == 'QB')).select(['full_name', 'position', 'team', 'draft_number']))
print(roster)

In [ ]:
players = nfl.load_players()
players = players.select(['gsis_id', 'pfr_id'])

snap = snap.join(players, left_on='pfr_player_id', right_on='pfr_id', how='left')

In [ ]:
print(df.select('player_id').head(5))

In [ ]:
print(snap.filter(pl.col('player') == 'Aaron Rodgers').select('gsis_id').head(5))

My goal is to take the last 3 seasons, then I need to only take the required values that I need to use for calculations or for making predictions, then I need to for each player have a method that will calculate how much that player would have scored in our fantasy league.  I will then append the scored value onto the edited dataset to 

values I need
Passing:
Passing Yards
TD Passes
40+ passes td
50+ passes td
Ints thrown
2pc thrown

Rushing:
Rushing Yards
Rush TD
50+ rush td
2p rush conversion

Receiving:
Receiving yards
Rec TD
40+ rec td
50+ rec td
2pt rec conv

Kicking:
PAT
PAT missed
total FG missed
FG made from distance 0-39
40-49, 
50-59, 
60+
fg missed distance 0-39, 
40-49

Team Defense:
number of sacks
int returned to td
fumble returned for td
kickoff return td                 +6
punt return td
block kick or punt for td
block kick or punt
def gets int
def recovers fumble
safety
points allowed:
0
1-6
7-13
14-17
18-21
22-27
28-34
35-45
46+
yards allowed:
< 100
100-199
200-299
350-399
100-449
450-499
500-549
550+

DefensivePlayer:
Sacks
Block kick
each int
fumble rec
fumble forced
safety
assisted tackles
solo tackles
passes defensed

Punting:
punts inside 20
Punt Average:
44.0+
42-43.9
40.0-41.9


In [ ]:
# method to make my fantasy score for a player
df = df.with_columns(
    pl.when(pl.col('position').is_in(['QB', 'RB', 'WR', 'TE']))
    .then(
        (pl.col('passing_yards') // 25) * 2 + 
        pl.col('passing_tds') * 6 +
        (pl.col('rushing_yards') // 10)* 1 +
        pl.col('rushing_tds') * 6 +
        (pl.col('receiving_yards') // 10) * 1.5 +
        pl.col('receiving_tds') * 6 +
        pl.col('fumbles_lost_total') * -2 +
        pl.col('passing_interceptions') * -2 + 
        pl.col('passing_2pt_conversions') * 2 + 
        pl.col('rushing_2pt_conversions') * 2 +
        pl.col('receiving_2pt_conversions') * 2
    )
    .otherwise(pl.col('fantasy_points'))
    .alias('my_fantasy_points')
)

In [ ]:
print(df.head())

In [ ]:
with pl.Config(tbl_cols=-1, tbl_rows=-1):
    print(
        df.filter(pl.col('player_name') == 'D.Kincaid')
        .head(5)[['player_display_name', 'team', 'week', 'my_fantasy_points', 'passing_yards', 'rushing_yards', 'receiving_yards', 'passing_tds', 'rushing_tds', 'receiving_tds', 'passing_interceptions', 'fumbles_lost_total']]
    )

In [ ]:
import sys
sys.path.append('..')  # if your notebook is in notebooks/ and src is at project root
from src.data_ingestion import load_or_fetch_weekly_data

df = load_or_fetch_weekly_data([2023, 2024, 2025])

In [ ]:
print(df.shape)                      # rows/columns — sanity check on size
print(df.columns)                    # confirm every expected column is present
print(df.head(10))                   # eyeball a sample
print(df['my_fantasy_points'].null_count())   # should be 0 or very low
print(df['my_fantasy_points'].describe())     # check the range looks sane, no wild outliers

In [ ]:
row = df.filter((pl.col('player_name').str.contains('J.Jefferson')) & (pl.col('week') == 6) & (pl.col('team') == 'MIN'))
row.glimpse()  # prints every column and its value for this row, easy to scan for nulls

In [ ]:
import requests
url = "https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_week_2023.parquet"
r = requests.get(url, timeout=30)
print(r.status_code, len(r.content))

In [ ]:
df = load_or_fetch_weekly_data([2023, 2024, 2025])

In [ ]:
raw = pl.read_parquet("../data/weekly_data.parquet")  # your saved output from clean_weekly_data
print(raw.filter(pl.col('player_id').is_null()).shape)

In [ ]:
data = pl.read_parquet("../data/features.parquet")  # your saved output from clean_weekly_data
with pl.Config(tbl_cols=-1, tbl_rows=-1):
    print(
        data.filter(pl.col('player_name') == 'D.Kincaid')
        .head(10)[['player_name', 'team', 'week', 'my_fantasy_points', 'passing_yards', 'rushing_yards', 'receiving_yards', 'passing_tds', 'rushing_tds', 'receiving_tds', 'passing_interceptions', 'fumbles_lost_total', 'offense_snaps', 'defense_snaps', 'st_snaps', 'offense_pct', 'defense_pct', 'st_pct']]
    )

In [ ]:
data = pl.read_parquet("../data/features.parquet")  # your saved output from clean_weekly_data
print(data.shape)
print(len(data.filter((pl.col('offense_snaps') > 0) | (pl.col('defense_snaps') > 0) | (pl.col('st_snaps') > 0))))

In [ ]:
snap = nfl.load_snap_counts([2023, 2024, 2025])
players = nfl.load_players().select(['gsis_id', 'pfr_id'])
snap_bridged = snap.join(players, left_on='pfr_player_id', right_on='pfr_id', how='left')

print(snap_bridged.shape[0])
print(snap_bridged.filter(pl.col('gsis_id').is_null()).shape[0])

In [ ]:
df_test = df.join(snap, left_on=['player_id', 'game_id'], right_on=['gsis_id', 'game_id'], how='left', suffix='_snap')
print(df_test.shape[0])
print(df_test.filter(pl.col('offense_snaps').is_not_null()).shape[0])

In [ ]:
snap = nfl.load_snap_counts([2023, 2024, 2025])
players = nfl.load_players().select(['gsis_id', 'pfr_id'])
snap_bridged = snap.join(players, left_on='pfr_player_id', right_on='pfr_id', how='left')

print("snap rows:", snap_bridged.shape[0])
print("gsis_id matched:", snap_bridged.filter(pl.col('gsis_id').is_not_null()).shape[0])

In [ ]:
# spot check a player you know had real snaps in a specific week
data.filter((pl.col('player_name').str.contains('A.Rodgers')) & (pl.col('week') == 8)).select('player_name', 'week', 'offense_snaps', 'offense_pct')

In [ ]:
print(data.columns)

In [ ]:
import sys
sys.path.append('..')
from src.models import *



In [ ]:
features = pl.read_parquet("../data/features.parquet")  # your saved output from clean_weekly_data

In [ ]:
df = encode_categoricals(features)
train_df, test_df = chronological_split(df, 2025, 4)

In [ ]:
print(test_df.columns)

In [ ]:
print(df['my_fantasy_points'].dtype)

In [ ]:
feature_column = get_feature_columns(train_df)
baseline_model = train_baseline(train_df)


In [ ]:
model = train_Linear_model(train_df, feature_column)

In [ ]:
predictions = predict(model, test_df, feature_column)

In [ ]:
print(evaluate(predictions['predicted_points'], test_df['my_fantasy_points']))

In [ ]:
model2 = train_tree_model(train_df, feature_column)

In [ ]:
predictions2 = predict(model2, test_df, feature_column)

In [ ]:
print(evaluate(predictions2['predicted_points'], test_df['my_fantasy_points']))

In [ ]:
print(evaluate(test_df['avg_points_last_3'], test_df['my_fantasy_points']))

In [ ]:
importances = pd.Series(model2.feature_importances_, index=feature_column).sort_values(ascending=False)
print(importances.head(10))

In [ ]:
for pos in ['QB', 'RB', 'WR', 'TE']:
    mask = test_df[f'position'{pos}] == pos  # adjust if position got dummy-encoded already
    pos_actual = test_df.filter(pl.col('position') == pos)['my_fantasy_points']
    pos_pred = predictions.filter(pl.col('position') == pos)['predicted_points']  # match rows for this position
    print(pos, evaluate(pos_pred, pos_actual))

In [ ]:
raw_test = pl.read_parquet("../data/features.parquet").filter(
    (pl.col("season") > 2025) | ((pl.col("season") == 2025) & (pl.col("week") >= 4))
)  # match the same split condition, on the pre-encoded data

comparison = predictions.join(
    raw_test.select(['player_id', 'season', 'week', 'position', 'my_fantasy_points']),
    on=['player_id', 'season', 'week']
)

for pos in ['QB', 'RB', 'WR', 'TE', 'K', 'P']:
    subset = comparison.filter(pl.col('position') == pos)
    print(pos, evaluate(subset['predicted_points'], subset['my_fantasy_points']))

In [ ]:
columns = train_df.columns
print(columns)
positions = train_df.select(pl.selectors.starts_with("position")).columns
print(positions)
for pos in positions:
    mask = test_df['']

# print(positions)
